# IMU Axis Alignment + DANN — 5-Channel Pipeline (preprocessed/)`baseline/baseline_train.py` 와 동일한 **5채널 전처리 데이터** 위에서 IMU(축 X/Y/Z) 정렬과 DANN end-to-end 를 적용한다.채널 구성 (`preprocessed/X_*.npy` shape `(N, 5000, 5)`):- ch 0 : `biceps`        — EMG (20-450Hz bandpass)- ch 1 : `triceps`       — EMG- ch 2-4 : `triceps_X/Y/Z` — IMU **(정렬 대상)**세션 단위 8:2 split, 1000Hz 리샘플, 세션 단위 z-score 모두 적용된 상태.## 비교 시나리오1. **Permutation + Sign** brute-force (IMU 3축 ↔ 48 후보)2. **PCA** 기반 글로벌 IMU 정렬3. **STranGAN** — IMU 윈도우 adversarial 학습4. **TCN baseline** (`AdvancedBaselineModel`, source-only) → 4가지 정렬로 target val 평가5. **STB → TCN + DANN** end-to-end joint 학습모든 정렬은 EMG 채널 (ch 0, 1) 을 건드리지 않고 **IMU 채널 (2:5) 에만 적용**.

In [ ]:
import os, sys, time, math, random, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import spectral_norm
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# project root 가 sys.path 에 있어야 baseline import 가능
sys.path.append(os.path.abspath('..'))
from baseline.baseline_model import AdvancedBaselineModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 1. Preprocessed 5채널 데이터 로드

In [ ]:
DATA_DIR = '../preprocessed_original'   # samsung2_original 기반 (YXZ 재라벨링 안 됨)
# 참고: '../preprocessed' 는 samsung2 (YXZ 미리 정렬됨) 로부터 생성된 것이라
#       축 정렬 비교 의미 없음. samsung2_original 을 쓰려면 먼저
#       `python data_preprocess_original.py` 한 번 실행해서 ../preprocessed_original/ 생성.

X_train         = np.load(f'{DATA_DIR}/X_train.npy').astype(np.float32)
y_train_raw     = np.load(f'{DATA_DIR}/y_train.npy')
X_val           = np.load(f'{DATA_DIR}/X_val.npy').astype(np.float32)
y_val_raw       = np.load(f'{DATA_DIR}/y_val.npy')
X_tgt_train     = np.load(f'{DATA_DIR}/X_target_train.npy').astype(np.float32)
y_tgt_train_raw = np.load(f'{DATA_DIR}/y_target_train.npy')
X_tgt_val       = np.load(f'{DATA_DIR}/X_target_val.npy').astype(np.float32)
y_tgt_val_raw   = np.load(f'{DATA_DIR}/y_target_val.npy')

le = LabelEncoder()
all_labels = np.concatenate([y_train_raw, y_val_raw, y_tgt_train_raw, y_tgt_val_raw])
le.fit(all_labels)

y_train     = le.transform(y_train_raw)
y_val       = le.transform(y_val_raw)
y_tgt_train = le.transform(y_tgt_train_raw)
y_tgt_val   = le.transform(y_tgt_val_raw)

CLASS_NAMES = list(le.classes_)
N_CLASSES   = len(CLASS_NAMES)
CH_NAMES    = ['biceps', 'triceps_EMG', 'triceps_X', 'triceps_Y', 'triceps_Z']
IMU_IDX     = slice(2, 5)
EMG_IDX     = slice(0, 2)
AXIS_NAME   = ['X', 'Y', 'Z']

print(f'DATA_DIR: {DATA_DIR}')
print(f'Source train: {X_train.shape}   val: {X_val.shape}')
print(f'Target train: {X_tgt_train.shape}   val: {X_tgt_val.shape}')
print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')

## 2. 공통 유틸 — DTW, IMU 전용 변환

In [ ]:
def _resample_mat(X, n):
    return X[np.linspace(0, len(X) - 1, n, dtype=int)]


def _dtw_raw(A, B, n_pts=200):
    """multivariate DTW. A,B: (T, C)."""
    a, b = _resample_mat(A, n_pts), _resample_mat(B, n_pts)
    D = np.full((n_pts + 1, n_pts + 1), np.inf); D[0, 0] = 0.0
    for i in range(1, n_pts + 1):
        for j in range(1, n_pts + 1):
            D[i, j] = np.linalg.norm(a[i-1] - b[j-1]) + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    return float(D[n_pts, n_pts])


def apply_W_to_data(X, W):
    """X: (N, T, 5) → IMU 채널에만 3x3 W 곱."""
    X_out = X.copy()
    X_out[:, :, IMU_IDX] = X_out[:, :, IMU_IDX] @ W
    return X_out


def per_class_first(X, y, idx=0):
    out = {}
    for cls in range(N_CLASSES):
        ids = np.where(y == cls)[0]
        if len(ids) > idx:
            out[cls] = X[ids[idx]]
    return out


print('utils ready.')

## 3. 축 정렬 방법론

### 3.1 Permutation + Sign brute-force (IMU 3축)운동별 (rep0 source, rep0 target) IMU 페어에 대해 48 후보의 mean DTW 계산 → ranking.

In [ ]:
PERMS = list(itertools.permutations([0, 1, 2]))
SIGNS = list(itertools.product([1, -1], repeat=3))


def perm_sign_W(perm, sign):
    W = np.zeros((3, 3), dtype=np.float32)
    for out_axis, in_axis in enumerate(perm):
        W[in_axis, out_axis] = sign[out_axis]
    return W


def _label(perm, sign):
    return (''.join(AXIS_NAME[i] for i in perm),
            ''.join('+' if v == 1 else '-' for v in sign))


# 운동별 대표 IMU 시퀀스
src_imu_rep = {cls: arr[:, IMU_IDX] for cls, arr in per_class_first(X_train,     y_train    ).items()}
tgt_imu_rep = {cls: arr[:, IMU_IDX] for cls, arr in per_class_first(X_tgt_train, y_tgt_train).items()}


def rank_perm_sign(n_pts=200):
    records = []
    for perm in PERMS:
        for sign in SIGNS:
            W = perm_sign_W(perm, sign)
            dtws = []
            for cls in src_imu_rep:
                if cls not in tgt_imu_rep: continue
                dtws.append(_dtw_raw(src_imu_rep[cls], tgt_imu_rep[cls] @ W, n_pts))
            if not dtws: continue
            p, s = _label(perm, sign)
            records.append({'perm': p, 'sign': s,
                            'mean_dtw': float(np.mean(dtws)),
                            '_perm': perm, '_sign': sign})
    df = pd.DataFrame(records).sort_values('mean_dtw').reset_index(drop=True)
    df.insert(0, 'rank', df.index + 1)
    return df


rank_df = rank_perm_sign()

print('=== Top 10 (perm, sign) by mean DTW ===')
print(rank_df[['rank', 'perm', 'sign', 'mean_dtw']].head(10).to_string(index=False))

print('\n=== YXZ 8개 sign 조합 ===')
print(rank_df[rank_df['perm'] == 'YXZ'][['rank', 'perm', 'sign', 'mean_dtw']].to_string(index=False))

top = rank_df.iloc[0]
best_perm, best_sign = top['_perm'], top['_sign']
W_perm = perm_sign_W(best_perm, best_sign)
print(f'\n채택: perm={top["perm"]}  sign={top["sign"]}  mean DTW = {top["mean_dtw"]:.2f}')
print('W_perm =\n', W_perm)

### 3.2 PCA-based (IMU stack → V_t.T @ V_s)

In [ ]:
def global_pca_W(n_pts=500, k_per_class=5):
    S_all, T_all = [], []
    for cls in range(N_CLASSES):
        s_idx = np.where(y_train     == cls)[0][:k_per_class]
        t_idx = np.where(y_tgt_train == cls)[0][:k_per_class]
        for i in s_idx:
            S_all.append(_resample_mat(X_train[i, :, IMU_IDX], n_pts))
        for i in t_idx:
            T_all.append(_resample_mat(X_tgt_train[i, :, IMU_IDX], n_pts))
    S = np.concatenate(S_all, 0); T = np.concatenate(T_all, 0)
    V_s = PCA(n_components=3).fit(S).components_
    V_t = PCA(n_components=3).fit(T).components_
    W   = (V_t.T @ V_s).astype(np.float32)
    aligned = T @ W
    for j in range(3):
        if np.corrcoef(S[:, j], aligned[:, j])[0, 1] < 0:
            W[:, j] *= -1
    return W


W_pca = global_pca_W()
print('W_pca =\n', np.round(W_pca, 3))

### 3.3 STranGAN (IMU 3채널 adversarial)source/target 의 IMU 윈도우만 떼서 `SpatialTransformerBlock` 학습.inference 시엔 5채널 입력에서 IMU 부분만 G 통과시키고 EMG 는 그대로 둠.

In [ ]:
class MinibatchDiscrimination1d(nn.Module):
    def __init__(self, in_features, out_features, kernel_dims=16):
        super().__init__()
        self.out_features = out_features
        self.kernel_dims  = kernel_dims
        self.T = nn.Parameter(torch.randn(in_features, out_features * kernel_dims) * 0.1)
    def forward(self, x):
        M = (x @ self.T).view(-1, self.out_features, self.kernel_dims)
        l1 = (M.unsqueeze(0) - M.unsqueeze(1)).abs().sum(-1)
        feats = torch.exp(-l1).sum(1)
        return torch.cat([x, feats], dim=1)


class LocalizationNet(nn.Module):
    def __init__(self, n_channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(9, 1), stride=(2, 1)), nn.SELU(),
            nn.MaxPool2d(kernel_size=(2, 1)),
            nn.Conv2d(32, 64, kernel_size=(9, 1), stride=(2, 1)), nn.SELU(),
            nn.AdaptiveAvgPool2d((4, n_channels)),
        )
        self.flat_dim = 64 * 4 * n_channels
    def forward(self, x):
        return self.net(x).flatten(1)


class SpatialTransformerBlock(nn.Module):
    def __init__(self, n_channels=3):
        super().__init__()
        self.loc = LocalizationNet(n_channels)
        self.fc_loc = nn.Sequential(
            nn.Linear(self.loc.flat_dim, 128), nn.SELU(),
            nn.Linear(128, 64),                nn.SELU(),
            nn.Linear(64, 12),
        )
        self.fc_loc[-1].weight.data.zero_()
        self.fc_loc[-1].bias.data.copy_(torch.tensor(
            [1,0,0, 0,1,0, 0,0,1, 0,0,0], dtype=torch.float32))
    def forward(self, x):
        x_btc = x.transpose(1, 2)
        feat  = self.loc(x_btc.unsqueeze(1))
        theta = self.fc_loc(feat).view(-1, 4, 3)
        ones  = torch.ones(x_btc.size(0), x_btc.size(1), 1, device=x.device, dtype=x.dtype)
        y     = torch.matmul(torch.cat([x_btc, ones], dim=2), theta)
        return y.transpose(1, 2), theta


class StranganDiscriminator(nn.Module):
    def __init__(self, n_channels=3):
        super().__init__()
        self.conv = nn.Sequential(
            spectral_norm(nn.Conv1d(n_channels, 64, 9, stride=2)), nn.SELU(),
            spectral_norm(nn.Conv1d(64, 128, 9, stride=2)),       nn.SELU(),
            spectral_norm(nn.Conv1d(128, 256, 9, stride=2)),      nn.SELU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.fc  = spectral_norm(nn.Linear(256 * 4, 64))
        self.mbd = MinibatchDiscrimination1d(64, 32, kernel_dims=16)
        self.out = spectral_norm(nn.Linear(64 + 32, 1))
    def forward(self, x):
        h = self.conv(x).flatten(1)
        h = F.selu(self.fc(h))
        h = self.mbd(h)
        return self.out(h)


print('STranGAN modules ready.')

In [ ]:
class TensorDS(Dataset):
    def __init__(self, X): self.X = torch.from_numpy(X)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]


def _cycle(loader):
    while True:
        for b in loader: yield b


def extract_imu_btct(X, subsample=None, seed=SEED):
    """X: (N, T, 5) → (N', 3, T) IMU only. subsample 으로 메모리/속도 제어."""
    if subsample is not None and subsample < len(X):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X), subsample, replace=False)
        X = X[idx]
    return X[:, :, IMU_IDX].transpose(0, 2, 1).astype(np.float32)


def train_strangan(X_src_imu, X_tgt_imu, epochs=20, gamma_rec=5.0, lr=2e-4, batch=64):
    src_loader = DataLoader(TensorDS(X_src_imu), batch_size=batch, shuffle=True, drop_last=True)
    tgt_loader = DataLoader(TensorDS(X_tgt_imu), batch_size=batch, shuffle=True, drop_last=True)

    G = SpatialTransformerBlock().to(DEVICE)
    D = StranganDiscriminator().to(DEVICE)
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    adv   = nn.BCEWithLogitsLoss(); rec = nn.SmoothL1Loss()

    iters = max(len(src_loader), len(tgt_loader))
    src_it, tgt_it = _cycle(src_loader), _cycle(tgt_loader)
    hist = {'D': [], 'G_adv': [], 'G_rec': []}

    t0 = time.time()
    for ep in range(1, epochs + 1):
        d_sum = g_adv_sum = g_rec_sum = 0.0; nb = 0
        for _ in range(iters):
            x_s = next(src_it).to(DEVICE); x_t = next(tgt_it).to(DEVICE)
            B = min(x_s.size(0), x_t.size(0))
            x_s, x_t = x_s[:B], x_t[:B]
            ones = torch.ones(B, 1, device=DEVICE); zers = torch.zeros(B, 1, device=DEVICE)

            with torch.no_grad():
                x_tf, _ = G(x_t)
            loss_D = 0.5 * (adv(D(x_s), ones) + adv(D(x_tf), zers))
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            x_tg, _ = G(x_t); x_sg, _ = G(x_s)
            l_adv = adv(D(x_tg), ones); l_rec = rec(x_sg, x_s)
            loss_G = l_adv + gamma_rec * l_rec
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

            d_sum += loss_D.item(); g_adv_sum += l_adv.item(); g_rec_sum += l_rec.item(); nb += 1
        hist['D'].append(d_sum/nb); hist['G_adv'].append(g_adv_sum/nb); hist['G_rec'].append(g_rec_sum/nb)
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f'ep {ep:>2}/{epochs}  D={d_sum/nb:.3f}  G_adv={g_adv_sum/nb:.3f}  '
                  f'G_rec={g_rec_sum/nb:.3f}  ({time.time()-t0:.1f}s)')
    return G, hist


@torch.no_grad()
def apply_G_to_5ch(X, G, batch=64):
    """X: (N, T, 5) → IMU 부분에만 G 적용해서 (N, T, 5) 반환."""
    G.eval()
    X_out = X.copy()
    imu_in = X[:, :, IMU_IDX].transpose(0, 2, 1).astype(np.float32)
    out = []
    for i in range(0, len(imu_in), batch):
        xb = torch.from_numpy(imu_in[i:i+batch]).to(DEVICE)
        y, _ = G(xb)
        out.append(y.cpu().numpy())
    aligned = np.concatenate(out, 0)
    X_out[:, :, IMU_IDX] = aligned.transpose(0, 2, 1)
    return X_out


# 학습: 효율 위해 src/tgt 각 2000 윈도우 서브샘플
X_src_imu = extract_imu_btct(X_train,     subsample=2000)
X_tgt_imu = extract_imu_btct(X_tgt_train, subsample=2000)
print('STranGAN training data:', X_src_imu.shape, X_tgt_imu.shape)

G_strangan, hist_strangan = train_strangan(X_src_imu, X_tgt_imu, epochs=20)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist_strangan['D'], label='D'); ax[0].plot(hist_strangan['G_adv'], label='G_adv')
ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].set_title('adversarial')
ax[1].plot(hist_strangan['G_rec']); ax[1].set_xlabel('epoch'); ax[1].set_title('G_rec')
plt.tight_layout(); plt.show()

## 4. DTW 비교 (IMU 축 기준)

In [ ]:
ALIGN_METHODS = {
    'raw':      lambda X: X,
    'perm':     lambda X: apply_W_to_data(X, W_perm),
    'pca':      lambda X: apply_W_to_data(X, W_pca),
    'strangan': lambda X: apply_G_to_5ch(X, G_strangan),
}
METHOD_ORDER = list(ALIGN_METHODS.keys())
METHOD_COLOR = {'raw': 'gray', 'perm': 'steelblue', 'pca': 'seagreen', 'strangan': 'tomato'}


def dtw_table_imu(n_per_class=2, n_pts=200):
    # 모든 정렬 결과를 미리 캐시 (target subset에 대해서만)
    tgt_subset_idx = {cls: np.where(y_tgt_train == cls)[0][:n_per_class] for cls in range(N_CLASSES)}
    all_tgt_idx = np.unique(np.concatenate([v for v in tgt_subset_idx.values() if len(v) > 0]))
    X_tgt_subset = X_tgt_train[all_tgt_idx]
    aligned_cache = {name: fn(X_tgt_subset) for name, fn in ALIGN_METHODS.items()}
    idx_map = {orig: new for new, orig in enumerate(all_tgt_idx)}

    rows = []
    for cls in range(N_CLASSES):
        s_ids = np.where(y_train == cls)[0][:n_per_class]
        t_ids = tgt_subset_idx[cls]
        if len(s_ids) == 0 or len(t_ids) == 0: continue
        row = {'class': CLASS_NAMES[cls]}
        for name, aligned_X in aligned_cache.items():
            dtws = []
            for s in s_ids:
                s_imu = X_train[s, :, IMU_IDX]
                for t in t_ids:
                    t_imu = aligned_X[idx_map[t], :, IMU_IDX]
                    dtws.append(_dtw_raw(s_imu, t_imu, n_pts))
            row[name] = float(np.mean(dtws))
        rows.append(row)
    df = pd.DataFrame(rows)
    df.loc[len(df)] = {'class': 'MEAN', **{n: df[n].mean() for n in ALIGN_METHODS}}
    return df


dtw_df = dtw_table_imu(n_per_class=2)
print(dtw_df.round(2).to_string(index=False))

# mean DTW bar chart
means = dtw_df.iloc[-1].drop('class').astype(float)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(means.index, means.values, color=[METHOD_COLOR[n] for n in means.index])
ax.set_ylabel('mean DTW (IMU 축)'); ax.set_title('Cross-domain IMU DTW by alignment')
for i, v in enumerate(means.values):
    ax.text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

# per-class grouped
ex_only = dtw_df.iloc[:-1]
x = np.arange(len(ex_only)); w = 0.20
fig, ax = plt.subplots(figsize=(11, 3.8))
for i, n in enumerate(METHOD_ORDER):
    ax.bar(x + (i - 1.5) * w, ex_only[n], width=w, color=METHOD_COLOR[n], label=n)
ax.set_xticks(x); ax.set_xticklabels(ex_only['class'], rotation=30, ha='right')
ax.set_ylabel('DTW'); ax.legend(loc='upper right'); ax.set_title('Per-class IMU DTW')
plt.tight_layout(); plt.show()

## 5. 5채널 파형 시각화 (1 운동 = 1 figure)

In [ ]:
def _smooth(sig, w=20):
    if w <= 1 or len(sig) < w: return sig
    return np.convolve(sig, np.ones(w)/w, mode='same')


def plot_5ch_alignment(cls_list, n_pts=200, smooth_w=30):
    """각 운동에 대해 (S1 ref, S2 raw, S2 perm/pca/strangan) 5채널 비교."""
    ch_color = ['#444', '#888', 'steelblue', 'seagreen', 'tomato']
    for cls in cls_list:
        s_id = np.where(y_train == cls)[0][:1]
        t_id = np.where(y_tgt_train == cls)[0][:1]
        if len(s_id) == 0 or len(t_id) == 0: continue
        s_id, t_id = s_id[0], t_id[0]

        S = X_train[s_id]                                  # (5000, 5)
        T_raw = X_tgt_train[t_id]
        aligned = {name: fn(X_tgt_train[t_id:t_id+1])[0] for name, fn in ALIGN_METHODS.items()}
        # DTW (IMU only)
        s_imu = S[:, IMU_IDX]
        dtws = {name: _dtw_raw(s_imu, mat[:, IMU_IDX], n_pts) for name, mat in aligned.items()}

        panels = [('S1 reference', S)] + [(f'S2 {n}  IMU-DTW={dtws[n]:.1f}', aligned[n]) for n in METHOD_ORDER]
        fig, axes = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3.0))
        x = np.arange(S.shape[0]) / 1000.0  # seconds @ 1000Hz
        for ax, (title, mat) in zip(axes, panels):
            for k in range(5):
                ax.plot(x, _smooth(mat[:, k], smooth_w),
                        color=ch_color[k], lw=0.9, alpha=0.85, label=CH_NAMES[k])
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('time (s)'); ax.tick_params(labelsize=8)
            ax.set_ylim(-4, 4); ax.margins(x=0.01)
        axes[0].legend(fontsize=7, loc='upper right', ncol=2)
        fig.suptitle(f'class={CLASS_NAMES[cls]}', fontsize=11, y=1.05)
        plt.tight_layout(); plt.show()


plot_5ch_alignment(list(range(min(4, N_CLASSES))))

## 6. TCN baseline (`AdvancedBaselineModel`) — source only 학습`baseline_train.py` 와 동일 설정: AdamW(wd=1e-4), CosineAnnealingLR, 5채널 raw target.

In [ ]:
class NumpyDS(Dataset):
    """(N, T, C) numpy → (C, T) tensor batch."""
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return (self.X[i], self.y[i]) if self.y is not None else self.X[i]


def train_baseline(epochs=15, batch=64, lr=1e-3, wd=1e-4):
    tr_loader = DataLoader(NumpyDS(X_train, y_train), batch_size=batch, shuffle=True, drop_last=True)
    va_loader = DataLoader(NumpyDS(X_val,   y_val),   batch_size=batch)

    model = AdvancedBaselineModel(in_channels=5, num_classes=N_CLASSES).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss()

    best_state, best_val = None, 0.0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); tot = 0.0; nb = 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        sched.step()

        model.eval(); correct = total = 0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                correct += (model(xb).argmax(-1) == yb).sum().item(); total += yb.numel()
        val_acc = correct / total

        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        print(f'ep {ep:>2}/{epochs}  loss={tot/nb:.3f}  src_val={val_acc:.3f}  '
              f'best={best_val:.3f}  ({time.time()-t0:.1f}s)')

    model.load_state_dict(best_state)
    return model, best_val


baseline_model, best_src_val = train_baseline(epochs=15)
print(f'\n학습 완료. best source val: {best_src_val:.3f}')

## 7. 정렬 방법별 target val 정확도 비교

In [ ]:
@torch.no_grad()
def eval_classifier(model, X, y, batch=128):
    ld = DataLoader(NumpyDS(X, y), batch_size=batch)
    model.eval(); correct = total = 0
    for xb, yb in ld:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        correct += (model(xb).argmax(-1) == yb).sum().item(); total += yb.numel()
    return correct / total


print(f'Source val:                {best_src_val:.3f}\n')
print('--- Target val by alignment ---')
final_accs = {}
for name, fn in ALIGN_METHODS.items():
    X_tgt_aligned = fn(X_tgt_val)
    acc = eval_classifier(baseline_model, X_tgt_aligned, y_tgt_val)
    final_accs[name] = acc
    print(f'  {name:>9} : {acc:.3f}')

# bar chart
fig, ax = plt.subplots(figsize=(7, 3.8))
labels = ['src_val'] + [f'tgt_{n}' for n in METHOD_ORDER]
values = [best_src_val] + [final_accs[n] for n in METHOD_ORDER]
colors = ['black'] + [METHOD_COLOR[n] for n in METHOD_ORDER]
ax.bar(labels, values, color=colors)
ax.axhline(1/N_CLASSES, color='red', lw=1, ls='--', label=f'chance (1/{N_CLASSES})')
ax.set_ylabel('accuracy'); ax.set_ylim(0, 1)
ax.set_title('5-ch TCN: target accuracy by alignment')
for i, v in enumerate(values):
    ax.text(i, v + 0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 8. STB → TCN + DANN end-to-end (5채널)- **STB**: IMU 3채널에만 affine θ 학습 (EMG 그대로 통과)- **TCN**: 5채널 입력, 동일 `AdvancedBaselineModel`- **Domain D**: STB 출력 5채널 전체 보고 도메인 판별- λ schedule: DANN paper `2/(1+exp(-10p)) - 1`

In [ ]:
# === GRL ===
class _GRLFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad):
        return -ctx.lambda_ * grad, None


def grl(x, lambda_):
    return _GRLFn.apply(x, lambda_)


# === Domain Discriminator (5채널 시그널 직접) ===
class DomainDiscriminator5ch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(5, 64, 11, stride=5), nn.ReLU(),    # 5000 → 1000
            nn.Conv1d(64, 128, 9, stride=4), nn.ReLU(),   # 1000 → 250
            nn.Conv1d(128, 128, 9, stride=4), nn.ReLU(),  # 250  → 62
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)


# === 조합 모델 ===
class STB_TCN_DANN_5ch(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.stb  = SpatialTransformerBlock(n_channels=3)
        self.tcn  = AdvancedBaselineModel(in_channels=5, num_classes=n_classes)
        self.disc = DomainDiscriminator5ch()

    def forward(self, x, lambda_):
        # x: (B, 5, T)
        emg = x[:, :2, :]
        imu = x[:, 2:, :]
        imu_aligned, theta = self.stb(imu)
        z = torch.cat([emg, imu_aligned], dim=1)
        logits = self.tcn(z)
        dom    = self.disc(grl(z, lambda_))
        return logits, dom, theta


# === 학습 ===
def train_dann(epochs=15, batch=64, lr=1e-3, wd=1e-4):
    src_loader    = DataLoader(NumpyDS(X_train,     y_train    ), batch_size=batch, shuffle=True, drop_last=True)
    va_loader     = DataLoader(NumpyDS(X_val,       y_val      ), batch_size=batch)
    tgt_loader    = DataLoader(NumpyDS(X_tgt_train, y_tgt_train), batch_size=batch, shuffle=True, drop_last=True)
    tgt_va_loader = DataLoader(NumpyDS(X_tgt_val,   y_tgt_val  ), batch_size=batch)

    model = STB_TCN_DANN_5ch(N_CLASSES).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = CosineAnnealingLR(opt, T_max=epochs)
    ce    = nn.CrossEntropyLoss(); bce = nn.BCEWithLogitsLoss()

    iters       = max(len(src_loader), len(tgt_loader))
    total_steps = epochs * iters
    src_it = _cycle(src_loader); tgt_it = _cycle(tgt_loader)

    @torch.no_grad()
    def eval_acc(loader):
        model.eval(); correct = total = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits, _, _ = model(xb, 0.0)
            correct += (logits.argmax(-1) == yb).sum().item(); total += yb.numel()
        return correct / total

    hist = {'cls': [], 'dom': [], 'src_val': [], 'tgt_val': [], 'lambda': []}
    step = 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        cls_sum = dom_sum = 0.0; nb = 0
        for _ in range(iters):
            xs, ys = next(src_it); xt, _ = next(tgt_it)
            xs, ys, xt = xs.to(DEVICE), ys.to(DEVICE), xt.to(DEVICE)
            B = min(xs.size(0), xt.size(0))
            xs, ys, xt = xs[:B], ys[:B], xt[:B]

            p   = step / max(total_steps - 1, 1)
            lam = 2.0 / (1.0 + math.exp(-10 * p)) - 1.0

            cls_s, dom_s, _ = model(xs, lam)
            _,     dom_t, _ = model(xt, lam)

            ones = torch.ones(B, 1, device=DEVICE)
            zers = torch.zeros(B, 1, device=DEVICE)
            l_cls = ce(cls_s, ys)
            l_dom = 0.5 * (bce(dom_s, ones) + bce(dom_t, zers))
            loss  = l_cls + l_dom
            opt.zero_grad(); loss.backward(); opt.step()

            cls_sum += l_cls.item(); dom_sum += l_dom.item(); nb += 1; step += 1
        sched.step()

        src_v = eval_acc(va_loader); tgt_v = eval_acc(tgt_va_loader)
        hist['cls'].append(cls_sum/nb); hist['dom'].append(dom_sum/nb)
        hist['src_val'].append(src_v); hist['tgt_val'].append(tgt_v); hist['lambda'].append(lam)
        print(f'ep {ep:>2}/{epochs}  λ={lam:.2f}  cls={cls_sum/nb:.3f}  dom={dom_sum/nb:.3f}  '
              f'src_val={src_v:.3f}  tgt_val={tgt_v:.3f}  ({time.time()-t0:.1f}s)')

    return model, hist


dann_model, dann_hist = train_dann(epochs=15)


# === 시각화 ===
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
axes[0].plot(dann_hist['cls'], label='cls'); axes[0].plot(dann_hist['dom'], label='dom')
axes[0].set_xlabel('epoch'); axes[0].set_title('losses'); axes[0].legend()
axes[1].plot(dann_hist['src_val'], label='src val', color='black')
axes[1].plot(dann_hist['tgt_val'], label='tgt val', color='tomato')
axes[1].axhline(1/N_CLASSES, color='red', lw=1, ls='--', label='chance')
axes[1].set_xlabel('epoch'); axes[1].set_ylim(0, 1); axes[1].set_title('accuracy'); axes[1].legend()
axes[2].plot(dann_hist['lambda'], color='seagreen')
axes[2].set_xlabel('epoch'); axes[2].set_title('λ schedule')
plt.tight_layout(); plt.show()


# === 비교 표 ===
print('\n=== Target val accuracy 비교 ===')
print(f'  Section 6 baseline (no STB, raw)   : {final_accs["raw"]:.3f}')
print(f'  Section 7 baseline + perm align    : {final_accs["perm"]:.3f}')
print(f'  Section 7 baseline + PCA align     : {final_accs["pca"]:.3f}')
print(f'  Section 7 baseline + STranGAN      : {final_accs["strangan"]:.3f}')
print(f'  Section 8 STB→TCN+DANN end-to-end  : {dann_hist["tgt_val"][-1]:.3f}  ← NEW')
print(f'  (best target val during training)  : {max(dann_hist["tgt_val"]):.3f}')